# FaceLift Pipeline: 2D Face → 3D Gaussian Splat → Depth Map

| Step | Content | Description |
|------|---------|-------------|
| 0 | Environment Setup | Install dependencies, check GPU |
| 1 | Download Dataset | Kaggle Human Faces |
| 2 | Preprocess Images | Face detection + cropping to 512×512 |
| 3 | FaceLift Inference | 2D → 3D Gaussian Splat |
| 4 | Export Depth Maps | Splat → Depth Map + RGB |
| 5 | Build Dataset | Paired RGB + Depth with train/val/test splits |

## Step 0: Environment Setup & GPU Check

In [ ]:
# ============================================================
# Install all dependencies (run once per environment)
# Set SKIP_INSTALL = True after first successful run
# ============================================================
SKIP_INSTALL = True  # <-- packages already installed

if not SKIP_INSTALL:
    import subprocess, sys, os, glob
    def pip_install(*packages):
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + list(packages))
    pip_install('torch==2.4.0', 'torchvision==0.19.0', '--index-url', 'https://download.pytorch.org/whl/cu124')
    pip_install('transformers==4.44.2', 'diffusers[torch]==0.30.3', 'huggingface-hub', 'xformers==0.0.27.post2', 'accelerate==0.33.0')
    pip_install('Pillow', 'opencv-python', 'scikit-image', 'lpips==0.1.4')
    pip_install('facenet-pytorch', '--no-deps')
    pip_install('rembg', 'onnxruntime')
    pip_install('trimesh', 'plyfile', 'easydict', 'einops')
    pip_install('pyyaml', 'tqdm', 'kaggle', 'rich')
    print('All dependencies installed!')
else:
    print('SKIP_INSTALL=True, skipping. Set False to reinstall.')

In [ ]:
import os, sys, json, yaml, shutil, struct
import numpy as np
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, Tuple
import math
from PIL import Image

# Robustly locate PROJECT_ROOT (idempotent — safe to re-run after cwd has drifted).
# Search order: notebook file location (VS Code) -> cwd -> hardcoded fallback.
def _find_project_root():
    search_starts = []
    # 1) VS Code Jupyter exposes the notebook path here
    nb_path = globals().get('__vsc_ipynb_file__') or globals().get('__session__')
    if nb_path:
        search_starts.append(Path(nb_path).resolve().parent)
    # 2) classic Jupyter
    try:
        import ipynbname
        search_starts.append(Path(ipynbname.path()).resolve().parent)
    except Exception:
        pass
    # 3) current working dir
    search_starts.append(Path(os.path.abspath('')).resolve())
    # 4) hardcoded fallback for this project
    search_starts.append(Path(r'D:\zmm\facelift_pipeline'))
    search_starts.append(Path(r'D:\zmm\facelift_pipeline\FaceLift'))

    tried = []
    for start in search_starts:
        for cand in [start] + list(start.parents):
            tried.append(cand)
            if (cand / 'configs' / 'pipeline_config.yaml').exists():
                return cand
    raise RuntimeError(f'Could not find configs/pipeline_config.yaml. Tried: {tried}')

PROJECT_ROOT = _find_project_root()
os.chdir(PROJECT_ROOT)
print('Working directory:', os.getcwd())

# Load configuration from YAML file
config_file = Path('configs/pipeline_config.yaml')
if config_file.exists():
    with open(config_file, encoding='utf-8') as f:
        config = yaml.safe_load(f)
    print('Config loaded from: ' + str(config_file))
    print('Paths configured: raw_dir=' + str(config['paths'].get('dataset_raw')) + ', output_dir=' + str(config['paths'].get('splat_output')))
else:
    print('WARNING: config file not found at ' + str(config_file))


In [ ]:
from pathlib import Path
import torch

# Setup directories
RAW_DIR = Path(config['paths']['dataset_raw']).resolve()
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Device setup
if torch.cuda.is_available():
    device = torch.device('cuda')
    print('GPU available: ' + torch.cuda.get_device_name(0))
else:
    device = torch.device('cpu')
    print('Using CPU (no GPU available)')

print('RAW_DIR: ' + str(RAW_DIR))
print('Device: ' + str(device))

## Step 1: Download Dataset from Kaggle

In [ ]:
# Download or check raw images
# Users should either use Kaggle API or manually place images in RAW_DIR

image_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
raw_images = sorted([f for f in RAW_DIR.iterdir() if f.suffix.lower() in image_exts])

print('Raw images found: ' + str(len(raw_images)))

if len(raw_images) == 0:
    print('No images found in ' + str(RAW_DIR))
    print('Please download from Kaggle: https://www.kaggle.com/datasets/ashwingupta3012/human-faces')
else:
    for i, img in enumerate(raw_images[:5]):
        print('  ' + str(i+1) + '. ' + img.name)

## Step 2: Preprocess Faces (Face Detection + Cropping)

In [ ]:
# (User can add helper code here if needed)

In [ ]:
# Robust cell 9: rembg model cache on D:, loud errors, periodic flush
import os, sys, traceback

# === Redirect rembg model cache to D: drive (C: is full) ===
REMBG_HOME = (PROJECT_ROOT / '.rembg_models').resolve()
REMBG_HOME.mkdir(parents=True, exist_ok=True)
os.environ['U2NET_HOME'] = str(REMBG_HOME)
os.environ['REMBG_HOME'] = str(REMBG_HOME)
print(f'rembg model cache: {REMBG_HOME}')

FACELIFT_DIR = (PROJECT_ROOT / 'FaceLift').resolve()
if str(FACELIFT_DIR) not in sys.path:
    sys.path.insert(0, str(FACELIFT_DIR))

from rembg import new_session, remove
from utils_folder.face_utils import crop_face, FACE_DETECTOR, FACE_SIZE, FACE_CENTER
from tqdm.auto import tqdm

# Try in order of quality. Don't crash if a model is unavailable.
PREFERRED_MODELS = ['birefnet-portrait', 'birefnet-general', 'u2net_human_seg', 'u2net']
rembg_session = None
REMBG_MODEL = None
for m in PREFERRED_MODELS:
    try:
        print(f'Loading rembg model: {m} ...')
        rembg_session = new_session(m)
        REMBG_MODEL = m
        print(f'  -> using {m}')
        break
    except Exception as e:
        print(f'  failed: {type(e).__name__}: {e}')
if rembg_session is None:
    raise RuntimeError('No rembg model could be loaded.')

RAW_DIR = (PROJECT_ROOT / config['paths']['dataset_raw'].lstrip('./')).resolve()
PROCESSED_DIR = (PROJECT_ROOT / config['paths']['dataset_cropped'].lstrip('./')).resolve()
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f'RAW_DIR:       {RAW_DIR}')
print(f'PROCESSED_DIR: {PROCESSED_DIR}')

TARGET_SIZE = 512
BG_COLOR = (255, 255, 255)

import gc
MAX_INPUT_SIDE = 1024  # downsize huge images before feeding rembg, saves RAM

def _resize_if_huge(pil_img, max_side=MAX_INPUT_SIDE):
    w, h = pil_img.size
    m = max(w, h)
    if m > max_side:
        s = max_side / m
        return pil_img.resize((int(w * s), int(h * s)), Image.LANCZOS)
    return pil_img

def preprocess_one(img_path, max_side=MAX_INPUT_SIDE):
    pil_in = Image.open(img_path).convert('RGB')
    pil_in = _resize_if_huge(pil_in, max_side)
    rgba = remove(pil_in, session=rembg_session)
    if rgba.mode != 'RGBA':
        rgba = rgba.convert('RGBA')
    bg = Image.new('RGBA', rgba.size, BG_COLOR + (255,))
    composited = Image.alpha_composite(bg, rgba).convert('RGB')
    arr = np.array(composited)
    cropped, _ = crop_face(arr, FACE_DETECTOR, FACE_SIZE, FACE_CENTER, TARGET_SIZE, BG_COLOR)
    return cropped

raw_images = sorted([f for f in RAW_DIR.iterdir()
                     if f.suffix.lower() in config['preprocessing']['supported_formats']])
max_img = config['kaggle'].get('max_images')
if max_img:
    raw_images = raw_images[:max_img]

existing_stems = {p.stem for p in PROCESSED_DIR.glob('*.png')}
pending_raw = [f for f in raw_images if f'{f.stem}_face' not in existing_stems]

print(f'Raw images:        {len(raw_images)}')
print(f'Already processed: {len(existing_stems)}')
print(f'Pending:           {len(pending_raw)}')

if len(pending_raw) == 0:
    print('Nothing to do - all raw images already processed.')
else:
    success, fail = 0, 0
    error_log = []
    # Only fatal errors (disk full / IO) trigger abort. MemoryError is recoverable via downsize+gc.
    FATAL_TYPES = (OSError,)  # OSError covers disk-full / Errno 28
    consecutive_fatal = 0
    pbar = tqdm(pending_raw, desc=f'{REMBG_MODEL}+align')
    for i, img_path in enumerate(pbar):
        out_path = PROCESSED_DIR / f'{img_path.stem}_face.png'
        try:
            try:
                out = preprocess_one(img_path)
            except MemoryError:
                gc.collect()
                pbar.write(f'[OOM retry @ 768] {img_path.name}')
                try:
                    out = preprocess_one(img_path, max_side=768)
                except MemoryError:
                    gc.collect()
                    pbar.write(f'[OOM retry @ 512] {img_path.name}')
                    out = preprocess_one(img_path, max_side=512)
            out.save(out_path)
            if not out_path.exists() or out_path.stat().st_size == 0:
                raise IOError(f'save reported success but file missing/empty: {out_path}')
            success += 1
            consecutive_fatal = 0
        except Exception as e:
            fail += 1
            err = f'{img_path.name}: {type(e).__name__}: {e}'
            error_log.append(err)
            pbar.write(f'[FAIL {fail}] {err}')
            if isinstance(e, FATAL_TYPES) and ('No space' in str(e) or 'Errno 28' in str(e)):
                consecutive_fatal += 1
                if consecutive_fatal >= 3:
                    pbar.write('\n!! disk full — aborting to avoid losing progress.')
                    break
            else:
                consecutive_fatal = 0
        if (i + 1) % 50 == 0:
            pbar.write(f'  checkpoint @ {i+1}/{len(pending_raw)}: success={success} fail={fail}')
            gc.collect()
        # Progress checkpoint every 50 images
        if (i + 1) % 50 == 0:
            pbar.write(f'  checkpoint @ {i+1}/{len(pending_raw)}: success={success} fail={fail}')
    print(f'\nDone! Success: {success}, Failed: {fail}')
    if error_log:
        log_path = PROCESSED_DIR.parent / 'cell9_errors.log'
        log_path.write_text('\n'.join(error_log), encoding='utf-8')
        print(f'Error details written to: {log_path}')

# Ensure all images are RGB
for f in PROCESSED_DIR.glob('*.png'):
    img = Image.open(f)
    if img.mode != 'RGB':
        img.convert('RGB').save(f)
print(f'All {len(list(PROCESSED_DIR.glob("*.png")))} images verified as RGB.')


## Step 3: FaceLift Inference (2D → 3D Gaussian Splat)

In [ ]:
# Step 3 setup: paths and model availability check
FACELIFT_REPO = Path(config['paths']['facelift_repo']).resolve()
CHECKPOINT_DIR = FACELIFT_REPO / 'checkpoints'
SPLAT_DIR = Path(config['paths']['splat_output']).resolve()
SPLAT_DIR.mkdir(parents=True, exist_ok=True)

# Check if FaceLift models are available
mvdiff_ckpt = CHECKPOINT_DIR / 'mvdiffusion' / 'pipeckpts'
gslrm_ckpt = CHECKPOINT_DIR / 'gslrm' / 'ckpt_0000000000021125.pt'

FACELIFT_AVAILABLE = mvdiff_ckpt.exists() and gslrm_ckpt.exists()

print('FaceLift repo: ' + str(FACELIFT_REPO))
print('Models available: ' + str(FACELIFT_AVAILABLE))
if not FACELIFT_AVAILABLE:
    print('Checkpoints not found. Run FaceLift inference.py first.')

In [ ]:
# Step 3: FaceLift inference parameters and batch processing setup
# TEST_DIR = None: Use all preprocessed images

BATCH_SIZE = 100
TEST_DIR = None

# Re-resolve dirs from PROJECT_ROOT (don't trust stale kernel state)
PROCESSED_DIR = (PROJECT_ROOT / config['paths']['dataset_cropped'].lstrip('./')).resolve()
SPLAT_DIR = (PROJECT_ROOT / config['paths']['splat_output'].lstrip('./')).resolve()
SPLAT_DIR.mkdir(parents=True, exist_ok=True)

input_dir = PROCESSED_DIR
output_dir = SPLAT_DIR

print(f'Input dir:  {input_dir}')
print(f'Output dir: {output_dir}')

# Find preprocessed images to process
image_exts = {'.jpg', '.jpeg', '.png'}
input_images = sorted([f for f in input_dir.iterdir() if f.suffix.lower() in image_exts])

# Check for already processed splats
processed_count = 0
for img in input_images:
    ply_file = output_dir / img.stem / 'gaussians.ply'
    if ply_file.exists():
        processed_count += 1

pending = len(input_images) - processed_count

# Sanity check vs raw_faces
RAW_DIR_CHK = (PROJECT_ROOT / config['paths']['dataset_raw'].lstrip('./')).resolve()
raw_count = len([f for f in RAW_DIR_CHK.iterdir()
                 if f.suffix.lower() in config['preprocessing']['supported_formats']])

print(f'Raw images (data/raw_faces):       {raw_count}')
print(f'Cropped images (cropped_faces):    {len(input_images)}')
print(f'  -> still need cell 9 to process: {raw_count - len(input_images)}')
print(f'Already inferred (splats):         {processed_count}')
print(f'Pending inference:                 {pending}')
print(f'Batch size: {BATCH_SIZE}')

if pending > 0:
    print(f'Ready to process {pending} images in batches of {BATCH_SIZE}')
else:
    print('All cropped images already processed.')


In [ ]:
# Run FaceLift inference via subprocess (real-time output)
# Self-contained: re-resolves dirs and recomputes pending — does not trust kernel state.
import subprocess

# Re-resolve everything from PROJECT_ROOT
PROCESSED_DIR = (PROJECT_ROOT / config['paths']['dataset_cropped'].lstrip('./')).resolve()
SPLAT_DIR     = (PROJECT_ROOT / config['paths']['splat_output'].lstrip('./')).resolve()
SPLAT_DIR.mkdir(parents=True, exist_ok=True)
FACELIFT_REPO = (PROJECT_ROOT / config['paths']['facelift_repo'].lstrip('./')).resolve()

# Re-check FaceLift availability
mvdiff_ckpt = FACELIFT_REPO / 'checkpoints' / 'mvdiffusion' / 'pipeckpts'
gslrm_ckpt  = FACELIFT_REPO / 'checkpoints' / 'gslrm' / 'ckpt_0000000000021125.pt'
FACELIFT_AVAILABLE = mvdiff_ckpt.exists() and gslrm_ckpt.exists()
print(f'FaceLift repo:    {FACELIFT_REPO}')
print(f'Models available: {FACELIFT_AVAILABLE}')

# Recompute pending from disk (do not trust prior cells)
image_exts = {'.jpg', '.jpeg', '.png'}
input_images = sorted([f for f in PROCESSED_DIR.iterdir() if f.suffix.lower() in image_exts])
pending_imgs = [f for f in input_images if not (SPLAT_DIR / f.stem / 'gaussians.ply').exists()]
pending = len(pending_imgs)

print(f'Cropped images: {len(input_images)}')
print(f'Already done:   {len(input_images) - pending}')
print(f'Pending:        {pending}')

if not FACELIFT_AVAILABLE:
    print('Models not available — check FaceLift checkpoints.')
elif pending == 0:
    print('All cropped images already have splats.')
else:
    inference_script = str(FACELIFT_REPO / 'inference.py')
    inf_cfg = config['inference']
    print(f'Input:  {PROCESSED_DIR}')
    print(f'Output: {SPLAT_DIR}')
    print(f'Pending: {pending} images')
    print(f'Running FaceLift inference...\n')

    cmd = [
        sys.executable, '-u', inference_script,   # -u: unbuffered stdout
        '--input_dir', str(PROCESSED_DIR),
        '--output_dir', str(SPLAT_DIR),
        '--auto_crop',
        '--guidance_scale_2D', str(inf_cfg.get('guidance_scale_2D', 3.0)),
        '--step_2D', str(inf_cfg.get('step_2D', 50)),
    ]
    print('CMD:', ' '.join(cmd))
    print('(Loading mvdiffusion + gslrm checkpoints — this takes ~1-2 min with no output)\n', flush=True)

    env = os.environ.copy()
    env['SKIP_TURNTABLE'] = '1'
    env['PYTHONUNBUFFERED'] = '1'

    process = subprocess.Popen(
        cmd,
        cwd=str(FACELIFT_REPO),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding='utf-8',
        errors='replace',
        env=env,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end='', flush=True)

    process.wait()

    if process.returncode != 0:
        print(f'\nWarning: Return code {process.returncode}')

    ply_count = len(list(SPLAT_DIR.glob('**/gaussians.ply')))
    print(f'\nTotal Gaussian Splats now: {ply_count}')


## Step 4.5: Post-Processing (Alignment + Depth Norm + Smoothing + Cleanup)

**Key improvements:**
1. **Facial Landmark Alignment** — Mediapipe detects landmarks on original + rendered RGB, computes affine transform to pixel-align depth/normal/opacity with original image
2. **Artifact Cleanup** — Opacity-based masking + connected components to remove floating Gaussian splat debris
3. **Normal Smoothing** — Bilateral filter to remove 3DGS brush-stroke artifacts while preserving geometric edges
4. **Depth Renormalization** — Nose-tip-relative depth with fixed quantization range (no more per-sample min/max drift)
5. **Depth-Normal Consistency** — Physical validation: depth gradients must align with surface normals; flags bad samples

In [ ]:
# Cell 19 (repaired): Re-scan raw_faces dynamically and postprocess
# This replaces the previously corrupted cell. Re-run after adding new images to data/raw_faces.

from pathlib import Path

RAW_DIR = Path(config['paths']['dataset_raw']).resolve()
image_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
raw_images = sorted([f for f in RAW_DIR.iterdir() if f.suffix.lower() in image_exts])
print(f'Raw images currently in {RAW_DIR}: {len(raw_images)}')
for f in raw_images[:5]:
    print('  -', f.name)
if len(raw_images) > 5:
    print(f'  ... and {len(raw_images)-5} more')
